In [1]:
import pandas as pd
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from transformers import AutoTokenizer

df = pd.read_csv("../bandi_neo4j.csv") 

colonne_testo = [
    'section_budget', 
    'section_experts', 
    'section_finality', 
    'section_intervention', 
    'section_participants'
]

# Dizionario per collezionare i Document divisi per sezione
datasets_per_sezione = {
    col.replace("section_", "").capitalize(): [] 
    for col in colonne_testo
}

model_id = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
tokenizer = AutoTokenizer.from_pretrained(model_id)
text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer=tokenizer,
    chunk_size=250,
    chunk_overlap=50,
    separators=["\n\n", "\n", "; ", ". ", " ", ""]  # priorità a fine-lista/fine-frase prima di spezzare a spazio
)
print("Creazione dei dataset divisi per sezione...")

for index, row in df.iterrows():
    titolo = row.get('title', 'Titolo Sconosciuto')
    bando_id = str(row.get('bando_id', 'ID_Sconosciuto'))
    
    for colonna in colonne_testo:
        if pd.notna(row[colonna]):
            testo_sezione = str(row[colonna]).strip()
            
            if testo_sezione:
                nome_sezione = colonna.replace("section_", "").capitalize()
                frammenti_testo = text_splitter.split_text(testo_sezione)
                
                for idx, frammento in enumerate(frammenti_testo):
                    # Inseriamo il nome della sezione alla fine del titolo dopo il trattino
                    testo_arricchito = (
                        f"Titolo: {titolo} - {nome_sezione}\n"
                        f"Contenuto: {frammento}"
                    )
                    
                    chunk_id = f"{bando_id}_{nome_sezione}_{idx}"
                    
                    meta_chunk = {
                        "chunk_id": chunk_id,
                        "bando_id": bando_id,
                        "titolo_bando": titolo,
                        "sezione_testo": nome_sezione
                    }
                    
                    doc = Document(
                        page_content=testo_arricchito,
                        metadata=meta_chunk
                    )
                    
                    datasets_per_sezione[nome_sezione].append(doc)

# Report finale dei risultati per sezione
print("\n--- Riepilogo Dataset Generati ---")
for nome_sezione, docs in datasets_per_sezione.items():
    if docs:
        max_tokens = max(len(tokenizer.encode(doc.page_content)) for doc in docs)
        print(f"Sezione '{nome_sezione}': {len(docs)} chunk totali | Max token per chunk: {max_tokens}")
    else:
        print(f"Sezione '{nome_sezione}': 0 chunk trovati.")

c:\Users\Allocca_g\Documents\GRAPH_RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Creazione dei dataset divisi per sezione...


Token indices sequence length is longer than the specified maximum sequence length for this model (543 > 512). Running this sequence through the model will result in indexing errors



--- Riepilogo Dataset Generati ---
Sezione 'Budget': 1219 chunk totali | Max token per chunk: 291
Sezione 'Experts': 1137 chunk totali | Max token per chunk: 291
Sezione 'Finality': 1088 chunk totali | Max token per chunk: 302
Sezione 'Intervention': 1661 chunk totali | Max token per chunk: 301
Sezione 'Participants': 1307 chunk totali | Max token per chunk: 302


In [2]:
documents = datasets_per_sezione["Finality"]

In [3]:
import time
from tqdm import tqdm
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# ============================================================
# 1. Inizializzazione del Modello MPNet Multilingua
# ============================================================

print(f"Caricamento del modello {model_id} in memoria...")

hf_embeddings = HuggingFaceEmbeddings(
    model_name=model_id,
    model_kwargs={
        "device": "cpu"
    },
    encode_kwargs={
        "normalize_embeddings": True
    }
)
if hasattr(hf_embeddings, "client"):
    hf_embeddings.client.max_seq_length = 512
elif hasattr(hf_embeddings, "embedding_ctx"):
    hf_embeddings.embedding_ctx.max_seq_length = 512
elif hasattr(hf_embeddings, "_client"):
    hf_embeddings._client.max_seq_length = 512

# ============================================================
# 2. Configurazione dei Batch
# ============================================================
batch_size = 250  
totale_chunks = len(documents)

print(f"\nInizio vettorizzazione di {totale_chunks} documenti...")

# ============================================================
# 3. Creazione del Database (Primo Batch)
# ============================================================
primo_batch = documents[:batch_size]
vectorstore = FAISS.from_documents(primo_batch, hf_embeddings)

# ============================================================
# 4. Aggiunta Incrementale (Con Barra di Caricamento)
# ============================================================
for i in tqdm(range(batch_size, totale_chunks, batch_size), desc="Vettorizzazione", unit="batch"):
    batch_corrente = documents[i : i + batch_size]
    vectorstore.add_documents(batch_corrente)

# ============================================================
# 5. Salvataggio su Disco
# ============================================================
cartella_salvataggio = "faiss_index_bandi_first_level"
vectorstore.save_local(cartella_salvataggio)

print(f"\n✅ Operazione completata! Indice salvato in: '{cartella_salvataggio}'")

C:\Users\Allocca_g\AppData\Local\Temp\ipykernel_21704\1085771987.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Caricamento del modello sentence-transformers/paraphrase-multilingual-mpnet-base-v2 in memoria...

Inizio vettorizzazione di 1088 documenti...


KeyboardInterrupt: 

In [4]:
import numpy as np
from sklearn.decomposition import PCA
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

# ============================================================
# 1. Caricamento Indice FAISS e Vettori
# ============================================================
cartella_salvataggio = "faiss_index_bandi_first_level"
model_id = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"

print("Caricamento del modello di embedding...")
hf_embeddings = HuggingFaceEmbeddings(
    model_name=model_id,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

print(f"Caricamento dell'indice FAISS da '{cartella_salvataggio}'...")
vectorstore = FAISS.load_local(cartella_salvataggio, hf_embeddings, allow_dangerous_deserialization=True)

# Estrazione degli embedding salvati in FAISS
index_to_docstore_id = vectorstore.index_to_docstore_id
embeddings_list = []

for idx_int in index_to_docstore_id.keys():
    vector = vectorstore.index.reconstruct(int(idx_int))
    embeddings_list.append(vector)

X = np.array(embeddings_list)
print(f"Forma originale della matrice degli embedding: {X.shape}")


# ============================================================
# 2. Applicazione della PCA al 90% di Varianza Spiegata
# ============================================================
# Passando un float tra 0 e 1 (es. 0.90), scikit-learn seleziona automaticamente 
# il numero minimo di componenti necessarie per raggiungere quella percentuale di varianza.
pca = PCA(n_components=0.90, random_state=42)
X_pca = pca.fit_transform(X)

varianza_cumulativa_effettiva = np.sum(pca.explained_variance_ratio_) * 100

print(f"\n✅ PCA completata!")
print(f"  - Numero di componenti selezionate: {X_pca.shape[1]}")
print(f"  - Forma dei dati ridotti: {X_pca.shape}")
print(f"  - Varianza cumulativa spiegata effettiva: {varianza_cumulativa_effettiva:.2f}%")

Caricamento del modello di embedding...
Caricamento dell'indice FAISS da 'faiss_index_bandi_first_level'...
Forma originale della matrice degli embedding: (1088, 768)

✅ PCA completata!
  - Numero di componenti selezionate: 97
  - Forma dei dati ridotti: (1088, 97)
  - Varianza cumulativa spiegata effettiva: 90.05%


In [5]:
import numpy as np
from umap import UMAP

def global_cluster_embeddings(embeddings, dim=10, n_neighbors=None, metric="euclidean"):
    """
    Esegue la riduzione dimensionale globale usando UMAP.
    """
    if n_neighbors is None:
        # Auto-scala con la radice quadrata del numero di embedding (es. N=5283 -> ~72)
        n_neighbors = int((len(embeddings) - 1) ** 0.5)
        
    print(f"Esecuzione UMAP globale su {len(embeddings)} campioni...")
    print(f"Parametri -> n_neighbors: {n_neighbors}, n_components (dim): {dim}, metric: {metric}")
    
    reduced_embeddings = UMAP(
        n_neighbors=n_neighbors, 
        n_components=dim, 
        metric=metric,
        random_state=42
    ).fit_transform(embeddings)
    
    return reduced_embeddings

# ============================================================
# Applicazione pratica sulla matrice X_pca ottenuta prima
# ============================================================
# Usiamo i dati ridotti con la PCA (X_pca) e fissiamo dim=10 come da specifiche
X_umap_global = global_cluster_embeddings(X_pca, dim=10)

print(f"\n✅ UMAP globale completato!")
print(f"  - Forma della matrice ridotta finale: {X_umap_global.shape}")

Esecuzione UMAP globale su 1088 campioni...
Parametri -> n_neighbors: 32, n_components (dim): 10, metric: euclidean


c:\Users\Allocca_g\Documents\GRAPH_RAG\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(



✅ UMAP globale completato!
  - Forma della matrice ridotta finale: (1088, 10)


In [6]:
import numpy as np
from sklearn.mixture import GaussianMixture

def find_optimal_clusters_gmm(embeddings, max_k=15):
    """
    Trova il numero ottimale di cluster k per il GMM utilizzando il BIC.
    """
    best_bic = np.inf
    best_k = 1
    best_gmm = None
    
    # Limitiamo max_k al numero di campioni disponibili se sono inferiori a max_k
    max_k = min(max_k, len(embeddings))
    if max_k < 2:
        return 1, GaussianMixture(n_components=1, covariance_type='full', random_state=42).fit(embeddings)

    print(f"Ricerca del k ottimale (da 1 a {max_k}) tramite BIC...")
    
    for k in range(1, max_k + 1):
        # Inizializziamo il GMM con covariance_type='full' come da specifiche del paper RAPTOR
        gmm = GaussianMixture(n_components=k, covariance_type='full', random_state=42, init_params='kmeans')
        gmm.fit(embeddings)
        
        # Calcoliamo il BIC (più basso è, migliore è il modello)
        bic = gmm.bic(embeddings)
        
        if bic < best_bic:
            best_bic = bic
            best_k = k
            best_gmm = gmm

        print(f"  - k={k}: BIC={bic:.2f} (migliore finora: k={best_k}, BIC={best_bic:.2f})")
            
    print(f"✅ Trovato! Numero ottimale di cluster (k): {best_k} (con BIC: {best_bic:.2f})")
    return best_k, best_gmm

# ============================================================
# Applicazione sul risultato di UMAP globale (X_umap_global)
# ============================================================
# Eseguiamo la ricerca del k ottimale e otteniamo il modello addestrato
optimal_k, global_gmm = find_optimal_clusters_gmm(X_umap_global, max_k=80)

# Otteniamo le etichette dei cluster globali per ciascun documento
global_labels = global_gmm.predict(X_umap_global)

print(f"\nForma delle etichette globali: {global_labels.shape}")

Ricerca del k ottimale (da 1 a 80) tramite BIC...
  - k=1: BIC=-6963.66 (migliore finora: k=1, BIC=-6963.66)
  - k=2: BIC=-12940.17 (migliore finora: k=2, BIC=-12940.17)
  - k=3: BIC=-17184.38 (migliore finora: k=3, BIC=-17184.38)
  - k=4: BIC=-19062.17 (migliore finora: k=4, BIC=-19062.17)
  - k=5: BIC=-19987.73 (migliore finora: k=5, BIC=-19987.73)
  - k=6: BIC=-21032.67 (migliore finora: k=6, BIC=-21032.67)
  - k=7: BIC=-21312.78 (migliore finora: k=7, BIC=-21312.78)
  - k=8: BIC=-21414.13 (migliore finora: k=8, BIC=-21414.13)
  - k=9: BIC=-21952.46 (migliore finora: k=9, BIC=-21952.46)
  - k=10: BIC=-22444.17 (migliore finora: k=10, BIC=-22444.17)
  - k=11: BIC=-22465.42 (migliore finora: k=11, BIC=-22465.42)
  - k=12: BIC=-22624.72 (migliore finora: k=12, BIC=-22624.72)
  - k=13: BIC=-22968.02 (migliore finora: k=13, BIC=-22968.02)
  - k=14: BIC=-22782.82 (migliore finora: k=13, BIC=-22968.02)
  - k=15: BIC=-22775.10 (migliore finora: k=13, BIC=-22968.02)
  - k=16: BIC=-23306.92 (

In [7]:
import numpy as np

# 1. Recuperiamo la lista dei documenti direttamente dall'indice FAISS esistente (se non è in memoria)
index_to_docstore_id = vectorstore.index_to_docstore_id
docstore = vectorstore.docstore

documents_list = []
for idx_int in sorted(index_to_docstore_id.keys()):
    doc_id = index_to_docstore_id[idx_int]
    doc = docstore.search(doc_id)
    documents_list.append(doc)

# 2. Calcoliamo le probabilità di appartenenza con il GMM globale
probabilities = global_gmm.predict_proba(X_umap_global)

# 3. Applichiamo la soglia cumulativa (es. 90%)
soglia_cumulativa = 0.90  
global_clusters_chunks = {i: [] for i in range(optimal_k)}

for idx, probs in enumerate(probabilities):
    # Ordiniamo gli indici dei cluster in ordine decrescente di probabilità
    sorted_indices = np.argsort(probs)[::-1]
    sorted_probs = probs[sorted_indices]
    
    # Somma cumulativa delle probabilità
    cum_probs = np.cumsum(sorted_probs)
    
    # Numero di cluster necessari per coprire la soglia cumulativa
    n_cluster_da_prendere = np.searchsorted(cum_probs, soglia_cumulativa) + 1
    
    # Selezioniamo gli indici dei cluster effettivi
    cluster_assegnati = sorted_indices[:n_cluster_da_prendere]
    
    for c_id in cluster_assegnati:
        global_clusters_chunks[c_id].append(documents_list[idx])

# 4. Verificiamo i risultati finali
for c_id, docs in global_clusters_chunks.items():
    if len(docs) > 0:
        print(f"Cluster Globale {c_id}: {len(docs)} chunk assegnati")

Cluster Globale 0: 72 chunk assegnati
Cluster Globale 1: 122 chunk assegnati
Cluster Globale 2: 59 chunk assegnati
Cluster Globale 3: 63 chunk assegnati
Cluster Globale 4: 36 chunk assegnati
Cluster Globale 5: 36 chunk assegnati
Cluster Globale 6: 49 chunk assegnati
Cluster Globale 7: 106 chunk assegnati
Cluster Globale 8: 81 chunk assegnati
Cluster Globale 9: 88 chunk assegnati
Cluster Globale 10: 29 chunk assegnati
Cluster Globale 11: 91 chunk assegnati
Cluster Globale 12: 67 chunk assegnati
Cluster Globale 13: 30 chunk assegnati
Cluster Globale 14: 57 chunk assegnati
Cluster Globale 15: 39 chunk assegnati
Cluster Globale 16: 21 chunk assegnati
Cluster Globale 17: 56 chunk assegnati


In [8]:
from umap import UMAP

def local_cluster_embeddings(
    embeddings,
    dim=10,
    num_neighbors=10,
    metric="euclidean"
):
    """
    Riduzione dimensionale locale con UMAP.
    Parametri come nel paper RAPTOR.
    """

    reduced_embeddings = UMAP(
        n_neighbors=num_neighbors,
        n_components=dim,
        metric=metric,
        random_state=42
    ).fit_transform(embeddings)

    return reduced_embeddings

In [9]:
local_cluster_data = {}

for global_cluster_id in range(optimal_k):

    indices = [
        i for i, probs in enumerate(probabilities)
        if global_cluster_id in np.argsort(probs)[::-1][
            :np.searchsorted(np.cumsum(np.sort(probs)[::-1]), soglia_cumulativa) + 1
        ]
    ]

    if len(indices) < 2:
        continue

    X_local = X_pca[indices]

    X_umap_local = local_cluster_embeddings(
        X_local,
        dim=10,
        num_neighbors=10,     # fisso
        metric="euclidean"       # come RAPTOR
    )

    local_cluster_data[global_cluster_id] = {
        "indices": indices,
        "X_pca": X_local,
        "X_umap": X_umap_local
    }

    print(
        f"Cluster globale {global_cluster_id}: "
        f"{len(indices)} chunk -> UMAP locale {X_umap_local.shape}"
    )

Cluster globale 0: 72 chunk -> UMAP locale (72, 10)


c:\Users\Allocca_g\Documents\GRAPH_RAG\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\Allocca_g\Documents\GRAPH_RAG\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Cluster globale 1: 122 chunk -> UMAP locale (122, 10)
Cluster globale 2: 59 chunk -> UMAP locale (59, 10)
Cluster globale 3: 63 chunk -> UMAP locale (63, 10)
Cluster globale 4: 36 chunk -> UMAP locale (36, 10)


c:\Users\Allocca_g\Documents\GRAPH_RAG\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\Allocca_g\Documents\GRAPH_RAG\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\Allocca_g\Documents\GRAPH_RAG\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Cluster globale 5: 36 chunk -> UMAP locale (36, 10)


...


Cluster globale 9: 88 chunk -> UMAP locale (88, 10)
Cluster globale 10: 29 chunk -> UMAP locale (29, 10)


c:\Users\Allocca_g\Documents\GRAPH_RAG\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\Allocca_g\Documents\GRAPH_RAG\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\Allocca_g\Documents\GRAPH_RAG\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Cluster globale 11: 91 chunk -> UMAP locale (91, 10)
Cluster globale 12: 67 chunk -> UMAP locale (67, 10)
Cluster globale 13: 30 chunk -> UMAP locale (30, 10)


c:\Users\Allocca_g\Documents\GRAPH_RAG\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\Allocca_g\Documents\GRAPH_RAG\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\Allocca_g\Documents\GRAPH_RAG\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\Allocca_g\Documents\GRAPH_RAG\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Cluster globale 14: 57 chunk -> UMAP locale (57, 10)
Cluster globale 15: 39 chunk -> UMAP locale (39, 10)
Cluster globale 16: 21 chunk -> UMAP locale (21, 10)
Cluster globale 17: 56 chunk -> UMAP locale (56, 10)


In [10]:
local_topics = {}

for global_cluster_id, data in local_cluster_data.items():

    X_umap_local = data["X_umap"]

    print(f"\n=== Cluster globale {global_cluster_id} ===")

    # Ricerca del numero ottimale di cluster locali
    optimal_local_k, local_gmm = find_optimal_clusters_gmm(
        X_umap_local,
        max_k= int(len(X_umap_local)/2)
    )

    # Etichette hard
    local_labels = local_gmm.predict(X_umap_local)

    # Probabilità (soft clustering)
    local_probabilities = local_gmm.predict_proba(X_umap_local)

    local_topics[global_cluster_id] = {
        "indices": data["indices"],
        "X_umap": X_umap_local,
        "gmm": local_gmm,
        "n_topics": optimal_local_k,
        "labels": local_labels,
        "probabilities": local_probabilities,
    }

    print(
        f"Topic locali trovati: {optimal_local_k} "
        f"({len(local_labels)} chunk)"
    )


=== Cluster globale 0 ===
Ricerca del k ottimale (da 1 a 36) tramite BIC...
  - k=1: BIC=-438.07 (migliore finora: k=1, BIC=-438.07)
  - k=2: BIC=-597.54 (migliore finora: k=2, BIC=-597.54)
  - k=3: BIC=-637.75 (migliore finora: k=3, BIC=-637.75)
  - k=4: BIC=-634.04 (migliore finora: k=3, BIC=-637.75)
  - k=5: BIC=-762.43 (migliore finora: k=5, BIC=-762.43)
  - k=6: BIC=-786.34 (migliore finora: k=6, BIC=-786.34)
  - k=7: BIC=-901.58 (migliore finora: k=7, BIC=-901.58)
  - k=8: BIC=-990.45 (migliore finora: k=8, BIC=-990.45)
  - k=9: BIC=-1036.01 (migliore finora: k=9, BIC=-1036.01)
  - k=10: BIC=-1080.47 (migliore finora: k=10, BIC=-1080.47)
  - k=11: BIC=-1051.80 (migliore finora: k=10, BIC=-1080.47)
  - k=12: BIC=-965.46 (migliore finora: k=10, BIC=-1080.47)
  - k=13: BIC=-882.03 (migliore finora: k=10, BIC=-1080.47)
  - k=14: BIC=-826.04 (migliore finora: k=10, BIC=-1080.47)
  - k=15: BIC=-782.68 (migliore finora: k=10, BIC=-1080.47)
  - k=16: BIC=-676.40 (migliore finora: k=10, 

In [11]:
# =====================================================
# Soft assignment ai topic locali
# =====================================================

soglia_cumulativa = 0.90

local_clusters_chunks = {}

for global_cluster_id, data in local_topics.items():

    probabilities = data["probabilities"]
    indices_originali = data["indices"]
    n_topics = data["n_topics"]

    # Dizionario dei topic locali del cluster globale corrente
    topic_dict = {i: [] for i in range(n_topics)}

    for i, probs in enumerate(probabilities):

        # Ordine decrescente delle probabilità
        sorted_indices = np.argsort(probs)[::-1]
        sorted_probs = probs[sorted_indices]

        # Somma cumulativa
        cumulative = np.cumsum(sorted_probs)

        # Numero minimo di topic che raggiunge la soglia
        n_topics_to_take = np.searchsorted(cumulative, soglia_cumulativa) + 1

        selected_topics = sorted_indices[:n_topics_to_take]

        # Documento originale
        original_doc_idx = indices_originali[i]
        document = documents_list[original_doc_idx]

        for topic in selected_topics:
            topic_dict[topic].append(document)

    local_clusters_chunks[global_cluster_id] = topic_dict

In [12]:
for global_cluster_id, topics in local_clusters_chunks.items():

    print(f"\n===== Cluster globale {global_cluster_id} =====")

    for topic_id, docs in topics.items():

        if len(docs) > 0:
            print(f"Topic {topic_id}: {len(docs)} chunk")


===== Cluster globale 0 =====
Topic 0: 6 chunk
Topic 1: 7 chunk
Topic 2: 11 chunk
Topic 3: 10 chunk
Topic 4: 9 chunk
Topic 5: 7 chunk
Topic 6: 6 chunk
Topic 7: 7 chunk
Topic 8: 4 chunk
Topic 9: 5 chunk

===== Cluster globale 1 =====
Topic 0: 4 chunk
Topic 1: 12 chunk
Topic 2: 8 chunk
Topic 3: 5 chunk
Topic 4: 3 chunk
Topic 5: 6 chunk
Topic 6: 8 chunk
Topic 7: 9 chunk
Topic 8: 6 chunk
Topic 9: 5 chunk
Topic 10: 10 chunk
Topic 11: 6 chunk
Topic 12: 4 chunk
Topic 13: 6 chunk
Topic 14: 9 chunk
Topic 15: 7 chunk
Topic 16: 5 chunk
Topic 17: 9 chunk

===== Cluster globale 2 =====
Topic 0: 11 chunk
Topic 1: 8 chunk
Topic 2: 4 chunk
Topic 3: 10 chunk
Topic 4: 6 chunk
Topic 5: 5 chunk
Topic 6: 7 chunk
Topic 7: 8 chunk

===== Cluster globale 3 =====
Topic 0: 5 chunk
Topic 1: 8 chunk
Topic 2: 4 chunk
Topic 3: 7 chunk
Topic 4: 7 chunk
Topic 5: 9 chunk
Topic 6: 7 chunk
Topic 7: 4 chunk
Topic 8: 4 chunk
Topic 9: 4 chunk
Topic 10: 4 chunk

===== Cluster globale 4 =====
Topic 0: 5 chunk
Topic 1: 4 chu

In [13]:
import random

def ispeziona_cluster(local_clusters_chunks, n_cluster_globali=3, n_topic_per_cluster=3, n_chunk_per_topic=5, seed=42):
    """
    Stampa un campione di cluster/topic con i relativi chunk, mostrando
    titolo del bando + sezione + un estratto del contenuto, per valutare
    la coerenza tematica prima di passarli all'LLM per il summary.
    """
    random.seed(seed)
    
    cluster_ids = list(local_clusters_chunks.keys())
    campione_cluster = random.sample(cluster_ids, min(n_cluster_globali, len(cluster_ids)))
    
    for global_cluster_id in campione_cluster:
        topics = local_clusters_chunks[global_cluster_id]
        print(f"\n{'='*90}")
        print(f"CLUSTER GLOBALE {global_cluster_id} | Numero di topic: {len(topics)}")
        print(f"{'='*90}")
        
        topic_ids = [tid for tid, docs in topics.items() if len(docs) > 0]
        campione_topic = random.sample(topic_ids, min(n_topic_per_cluster, len(topic_ids)))
        
        for topic_id in campione_topic:
            docs = topics[topic_id]
            print(f"\n--- Topic {topic_id} | {len(docs)} chunk totali ---")
            
            campione_docs = random.sample(docs, min(n_chunk_per_topic, len(docs)))
            
            for i, doc in enumerate(campione_docs, 1):
                # Estrai contenuto e metadata in modo robusto (str, Document, dict)
                if isinstance(doc, str):
                    text_content = doc
                    metadata = {}
                elif hasattr(doc, "page_content"):
                    text_content = doc.page_content
                    metadata = getattr(doc, "metadata", {})
                elif isinstance(doc, dict):
                    text_content = doc.get("page_content") or doc.get("text", "")
                    metadata = doc.get("metadata", {})
                else:
                    text_content = str(doc)
                    metadata = {}
                
                titolo = metadata.get("titolo_bando", "N/D")
                sezione = metadata.get("sezione_testo", "N/D")
                bando_id = metadata.get("bando_id", "N/D")
                
                estratto = text_content.strip()[:200].replace("\n", " ")
                
                print(f"  [{i}] bando_id={bando_id} | sezione={sezione}")
                print(f"      titolo: {titolo}")
                print(f"      estratto: {estratto}...")
                print()


def controlla_coerenza_topic(local_clusters_chunks):
    """
    Per ogni topic, mostra QUANTI bandi diversi (bando_id unici) ci sono dentro.
    Se un topic ha pochi chunk ma molti bando_id diversi, potrebbe essere
    un segnale di clustering poco coerente (bandi non correlati raggruppati insieme).
    Se invece ha pochi bando_id ripetuti (stesse sezioni dello stesso bando),
    potrebbe essere un segnale opposto (poca diversità nel topic).
    """
    print(f"\n{'='*90}")
    print("COERENZA TOPIC: numero di bando_id distinti per topic")
    print(f"{'='*90}")
    
    for global_cluster_id, topics in local_clusters_chunks.items():
        for topic_id, docs in topics.items():
            if len(docs) == 0:
                continue
            
            bando_ids = set()
            for doc in docs:
                metadata = getattr(doc, "metadata", {}) if hasattr(doc, "metadata") else (doc.get("metadata", {}) if isinstance(doc, dict) else {})
                bando_ids.add(metadata.get("bando_id", "N/D"))
            
            n_chunk = len(docs)
            n_bandi = len(bando_ids)
            rapporto = n_chunk / n_bandi if n_bandi > 0 else 0
            
            print(f"Cluster {global_cluster_id} | Topic {topic_id}: {n_chunk} chunk, {n_bandi} bandi distinti (rapporto {rapporto:.1f})")


# Uso:
ispeziona_cluster(local_clusters_chunks, n_cluster_globali=3, n_topic_per_cluster=3, n_chunk_per_topic=5)
controlla_coerenza_topic(local_clusters_chunks)


CLUSTER GLOBALE 3 | Numero di topic: 11

--- Topic 3 | 7 chunk totali ---
  [1] bando_id=904 | sezione=Finality
      titolo: Horizon Europe – Cluster 5: bando Mobilità (HORIZON-CL5-2026-05)
      estratto: Titolo: Horizon Europe – Cluster 5: bando Mobilità (HORIZON-CL5-2026-05) - Finality Contenuto: La call Mobility riguarda diversi Topic della Destinazione 5 “ Soluzioni pulite e competitive per tutte l...

  [2] bando_id=675 | sezione=Finality
      titolo: Liguria - Bando FEAMPA - Investimenti per migliorare l'efficienza energetica e la mitigazione degli impatti sui cambiamenti climatici
      estratto: Titolo: Liguria - Bando FEAMPA - Investimenti per migliorare l'efficienza energetica e la mitigazione degli impatti sui cambiamenti climatici - Finality Contenuto: Il bando persegue l’obiettivo strate...

  [3] bando_id=903 | sezione=Finality
      titolo: Horizon Europe – Cluster 5: bando Mobilità (HORIZON-CL5-2026-06-two-stage)
      estratto: Titolo: Horizon Europe – Cluster 5: b

In [107]:
SYSTEM_PROMPT = """
Sei un assistente specializzato nella sintesi tematica di testi per sistemi RAG gerarchici (RAPTOR).
Il tuo compito NON è riassumere ogni frammento in sequenza, ma identificare i temi, i pattern e i meccanismi comuni che attraversano l'intero gruppo di frammenti, producendo una sintesi concettuale unificata.
Devi essere accurato e non aggiungere mai informazioni non supportate dal contenuto fornito.
Devi attenerti STRETTAMENTE al formato richiesto, senza mai aggiungere introduzioni, commenti, spiegazioni sul tuo operato o frasi di chiusura.
"""

USER_PROMPT_TEMPLATE = """
I seguenti frammenti di testo appartengono a un unico gruppo tematico di bandi/documenti relativi a finanziamenti pubblici.

IL TUO COMPITO:
Genera un titolo generale e un riassunto che catturino il TEMA COMUNE e i PATTERN RICORRENTI condivisi dai frammenti, non un elenco descrittivo bando per bando.

COME STRUTTURARE IL RIASSUNTO:
1. Individua gli elementi trasversali che accomunano i frammenti (beneficiari, meccanismo di incentivo, finalità).
2. Organizza il discorso attorno a QUESTI elementi trasversali, non attorno all'elenco delle singole regioni/enti/bandi.
3. Cita regioni, enti o bandi specifici solo come esempi a supporto di un pattern, non come struttura portante di ogni frase.
4. Se più bandi condividono lo stesso meccanismo, raggruppali in un'unica frase (es. "Diverse regioni, tra cui X, Y e Z, adottano un pagamento annuale per ettaro a compensazione di...") invece di descriverli uno per uno.
5. Il Contenuto deve essere un UNICO blocco di prosa continua e scorrevole. NON usare grassetto, sottotitoli, intestazioni di sezione, o qualsiasi altra suddivisione visiva in "capitoletti" o "aree tematiche" separate. NON usare elenchi puntati o numerati: anche se il gruppo copre più pattern distinti, collegali con connettivi logici (es. "Parallelamente...", "Un secondo meccanismo ricorrente è...") all'interno dello stesso flusso di prosa, non con punti elenco o titoli in grassetto.

COME CHIUDERE IL TESTO (regola più importante):
Il "Contenuto" deve terminare su un dettaglio concreto e specifico (una cifra, una scadenza, un meccanismo, un esempio puntuale) — MAI su una frase che riepiloga, generalizza o commenta l'insieme di quanto appena scritto.
Prima di scrivere l'ultima frase, chiediti: "sto aggiungendo un'informazione nuova o sto solo ripetendo/riassumendo con altre parole ciò che ho già detto?" Se la risposta è la seconda, elimina la frase e fermati alla frase precedente.
Frasi vietate in QUALSIASI riformulazione (indipendentemente dalle parole usate): "In sintesi...", "Tutti i bandi/programmi/interventi condividono/mirano a/convergono su...", "Questo dimostra/evidenzia/conferma che...", "Nel complesso...", o qualsiasi frase che descriva il riassunto stesso invece del contenuto dei bandi.

COSA EVITARE TASSATIVAMENTE:
- Elencare i frammenti nell'ordine in cui sono presentati: riorganizza sempre per tema, non per sequenza.
- Frasi che si limitano a giustapporre informazioni senza collegarle concettualmente.
- NON citare mai l'etichetta numerica del frammento originale (es. "Frammento 3", "Frammento 6", "vedi Frammento 9"): queste etichette sono solo un riferimento interno per te, servono a distinguere i blocchi di testo che ricevi, e NON devono MAI comparire nel testo del riassunto finale. Cita invece l'ente/regione/bando a cui il contenuto si riferisce, se rilevante.

COSA MANTENERE:
- I dettagli informativi realmente distintivi (cifre, soglie, requisiti, scadenze) quando servono a illustrare un pattern o una variazione significativa rispetto ad esso.
- L'accuratezza fattuale: non generalizzare in modo che perda precisione su dati verificabili.
- Ciò che rende QUESTO gruppo di bandi diverso da altri gruppi simili (es. non limitarti a dire "tutela della biodiversità" se il gruppo è specificamente su impollinatori, o su zone umide, o su razze autoctone: nomina l'oggetto specifico fin dal titolo).

REGOLE TASSATIVE SULL'OUTPUT:
- NON inserire MAI frasi di commento finale (vedi sopra).
- Termina il testo del "Contenuto" direttamente con l'ultimo punto fermo del paragrafo conclusivo, senza aggiungere nulla dopo.
- TITOLO: massimo 12 parole. Deve essere specifico e distintivo, non un'etichetta generica applicabile a decine di altri gruppi di bandi. Nessun grassetto o formattazione nel titolo.
- LUNGHEZZA MASSIMA DEL CONTENUTO: al massimo 300 parole — nessun sottotitolo, nessun grassetto, nessuna suddivisione in sezioni. Conta le parole mentre scrivi e fermati appena ti avvicini al limite: se stai per superarlo, taglia i dettagli meno distintivi invece di aggiungere altro testo o una frase di chiusura riassuntiva.

FORMATO RICHIESTO (rispondi ESCLUSIVAMENTE così):

Titolo: <titolo specifico e distintivo del cluster>

Contenuto: <riassunto unificato organizzato per pattern tematici, in prosa continua>

Testo completo dei frammenti da sintetizzare:
{context}
"""

In [ ]:
import os
import time
import json
import re
from transformers import AutoTokenizer
from google import genai
from google.genai import types
from google.genai.errors import ClientError

OUTPUT_FILE = "riassunti_raptor_progress.json"

# ============================================================
# Configurazione Gemini API - MULTI KEY
# ============================================================
# Meglio leggerle da variabili d'ambiente per non lasciarle in chiaro nel file.
# Ottieni le chiavi su https://aistudio.google.com/apikey
GEMINI_API_KEYS = [
    os.environ.get("GEMINI_API_KEY_1", "AQ.Ab8RN6LzERfVFSHpL1BB7JM-FfTxzvMPdlSs6HdLvZtig6h6Og")
    # aggiungi altre chiavi qui se ne hai altre
]
GEMINI_API_KEYS = [k for k in GEMINI_API_KEYS if k]

if not GEMINI_API_KEYS:
    raise ValueError("Nessuna API key valida trovata! Impostale in GEMINI_API_KEYS "
                      "(o nelle variabili d'ambiente GEMINI_API_KEY_1, GEMINI_API_KEY_2, ...).")

MODEL_NAME = "gemini-3.6-flash"

# ============================================================
# Gestore delle chiavi con rotazione e cooldown
# ============================================================
class KeyManager:
    def __init__(self, keys):
        self.keys = keys
        self.cooldown_until = {k: 0.0 for k in keys}
        self.current_index = 0
        self.clients = {k: genai.Client(api_key=k) for k in keys}

    def _available_keys(self):
        now = time.time()
        return [k for k in self.keys if self.cooldown_until[k] <= now]

    def mark_rate_limited(self, key, wait_seconds):
        self.cooldown_until[key] = time.time() + wait_seconds
        print(f"    🔒 Chiave ...{key[-6:]} in cooldown per {wait_seconds}s "
              f"(disponibile di nuovo alle {time.strftime('%H:%M:%S', time.localtime(self.cooldown_until[key]))}).")

    def get_next_key(self):
        """Ritorna una chiave utilizzabile subito, oppure None se tutte sono in cooldown."""
        available = self._available_keys()
        if not available:
            return None
        for i in range(len(self.keys)):
            idx = (self.current_index + i) % len(self.keys)
            k = self.keys[idx]
            if k in available:
                self.current_index = (idx + 1) % len(self.keys)
                return k
        return None

    def seconds_until_next_available(self):
        now = time.time()
        return max(0, min(self.cooldown_until[k] for k in self.keys) - now)

    def client_for(self, key):
        return self.clients[key]


key_manager = KeyManager(GEMINI_API_KEYS)

# ============================================================
# Tokenizer per il conteggio dei token (per il tuo controllo, non usato dalla API Gemini)
# ============================================================
TOKENIZER_ID = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID)


def count_tokens(text: str) -> int:
    return len(tokenizer.encode(text))


def load_progress(filepath):
    if os.path.exists(filepath):
        with open(filepath, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}


def save_progress(filepath, data):
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=4)


# ============================================================
# Parsing del tempo d'attesa dal messaggio di errore
# ============================================================
def parse_wait_seconds(error_str: str, default: int = 60) -> int:
    # Gemini di solito espone retryDelay tipo "retryDelay": "23s" nel JSON dell'errore 429
    match_retry_delay = re.search(r'retryDelay["\']?\s*:\s*["\']?(\d+(?:\.\d+)?)s', error_str)
    if match_retry_delay:
        return int(float(match_retry_delay.group(1))) + 5

    match_min = re.search(r'in\s+([0-9]+)m([0-9.]+)s', error_str)
    match_sec = re.search(r'in\s+([0-9.]+)s', error_str)

    if match_min:
        mins = float(match_min.group(1))
        secs = float(match_min.group(2))
        return int((mins * 60) + secs) + 10
    elif match_sec:
        secs = float(match_sec.group(1))
        return int(secs) + 10
    return default


# ============================================================
# Funzione di generazione con rotazione chiavi + retry
# ============================================================
def summarize_cluster_with_retry(context: str, max_retries: int = 20) -> str:
    prompt = USER_PROMPT_TEMPLATE.format(context=context)

    attempt = 0
    while attempt < max_retries:
        current_key = key_manager.get_next_key()

        if current_key is None:
            wait_s = key_manager.seconds_until_next_available()
            wait_s = max(wait_s, 1)
            print(f"    ⏳ Tutte le chiavi sono in rate limit. Aspetto {int(wait_s)}s "
                  f"finché una si libera...")
            time.sleep(wait_s)
            continue

        attempt += 1
        client = key_manager.client_for(current_key)

        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt,
                config=types.GenerateContentConfig(
                    system_instruction=SYSTEM_PROMPT,
                    temperature=0.0,
                ),
            )
            return response.text.strip()

        except ClientError as e:
            error_str = str(e)
            print(f"    [!] Errore con chiave ...{current_key[-6:]} "
                  f"(Tentativo {attempt}/{max_retries}): {error_str}")

            if "429" in error_str or "RESOURCE_EXHAUSTED" in error_str:
                wait_seconds = parse_wait_seconds(error_str, default=60)
                key_manager.mark_rate_limited(current_key, wait_seconds)
                continue
            else:
                if attempt < max_retries:
                    time.sleep(15)

        except Exception as e:
            print(f"    [!] Errore generico con chiave ...{current_key[-6:]} "
                  f"(Tentativo {attempt}/{max_retries}): {e}")
            if attempt < max_retries:
                time.sleep(15)

    return "ERRORE_GENERAZIONE: Impossibile generare la sintesi dopo i tentativi massimi."


def main():
    results = load_progress(OUTPUT_FILE)

    for global_cluster_id, topics in local_clusters_chunks.items():
        cluster_key = str(global_cluster_id)
        if cluster_key not in results:
            results[cluster_key] = {}

        print(f"\n===== Cluster globale {cluster_key} =====")

        for topic_id, docs in topics.items():
            if len(docs) == 0:
                continue

            topic_key = str(topic_id)

            if topic_key in results[cluster_key]:
                existing_summary = results[cluster_key][topic_key].get("summary", "")
                if "ERRORE_GENERAZIONE" not in existing_summary and existing_summary.strip():
                    print(f"--- ⏭️ Topic {topic_key} già completato (Skipped). ---")
                    continue

            print(f"\n--- ⏳ Elaborazione Topic {topic_key} ({len(docs)} chunk) ---")

            context_parts = []
            for i, doc in enumerate(docs, 1):
                text_content = doc if isinstance(doc, str) else (
                    doc.page_content if hasattr(doc, "page_content") else doc.get("page_content", "")
                )
                if text_content.strip():
                    context_parts.append(f"--- Frammento {i} ---\n{text_content.strip()}")

            context = "\n\n".join(context_parts)

            try:
                summary = summarize_cluster_with_retry(context=context)
                tokens_count = count_tokens(summary)

                print("\n--- RISULTATO FINALE ---")
                print(summary)
                print(f"\nToken effettivi: {tokens_count}")
                print("------------------------\n")

                results[cluster_key][topic_key] = {
                    "chunks_originali": len(docs),
                    "summary": summary,
                    "tokens": tokens_count
                }

                save_progress(OUTPUT_FILE, results)
                time.sleep(3)

            except Exception as e:
                print(f"❌ Errore fatale nel cluster {cluster_key} topic {topic_key}: {e}")

    print("\n✅ Elaborazione di tutti i cluster completata!")
    return results


if __name__ == "__main__":
    # Assicurati che il dizionario 'local_clusters_chunks' sia definito prima di lanciare il main
    summaries = main()


===== Cluster globale 0 =====

--- ⏳ Elaborazione Topic 0 (6 chunk) ---

--- RISULTATO FINALE ---
Titolo: Incentivi regionali per la sostenibilità agricola, biodiversità e tutela del territorio

Contenuto: Gli interventi di sviluppo rurale (SRA, SRB, SRD) adottati dalle diverse regioni condividono la finalità di coniugare la sostenibilità economica delle aziende agricole con la tutela ambientale, il contrasto alla perdita di biodiversità e la mitigazione dei cambiamenti climatici. Il meccanismo d'incentivo prevalente si basa sull'erogazione di pagamenti o premi annuali per ettaro di superficie agricola utilizzata, destinati a compensare i minori ricavi o i maggiori costi legati all'adozione di pratiche a basso impatto. Tale logica compensativa si applica alla conversione e al mantenimento dell'agricoltura biologica ai sensi del regolamento UE 2018/848 in Basilicata, dove il sostegno copre tutte le tipologie colturali, prati e pascoli con la sola esclusione dei terreni a riposo, così c

In [18]:
#create summaries by reading riassunti_raptor_progress.json 
import json
summaries = json.load(open("riassunti_raptor_progress.json", "r", encoding="utf-8"))

In [32]:
import re

def estrai_titolo_e_contenuto(testo: str) -> tuple[str, str]:
    """Estrae titolo e contenuto da una stringa formattata."""
    if not testo:
        return "", ""
        
    titolo = ""
    contenuto = testo.strip()
    
    # Estrazione del Titolo
    match_titolo = re.search(r"^Titolo:\s*(.*)", testo, re.MULTILINE)
    if match_titolo:
        titolo = match_titolo.group(1).strip()
    
    # Estrazione del Contenuto
    match_contenuto = re.search(r"Contenuto:\s*(.*)", testo, re.DOTALL)
    if match_contenuto:
        contenuto = match_contenuto.group(1).strip()
    elif match_titolo:
        contenuto = testo.replace(match_titolo.group(0), "").strip()
        
    return titolo, contenuto


def crea_dataset_grafo(documents, summaries, local_clusters_chunks):
    """
    Crea la lista di nodi per il grafo a partire dalle tre strutture:
    - documents: lista dei Document originali (foglie)
    - summaries: dizionario con i summary generati
    - local_clusters_chunks: dizionario di mapping (cluster -> topic -> lista di Document figli)
    """
    dataset = []
    processed_chunks = set()

    # ============================================================
    # 1. CREAZIONE NODI CHUNK (Foglie)
    # ============================================================
    print("🔹 Creazione nodi per i Chunk originali...")
    for doc in documents:
        chunk_id = doc.metadata.get("chunk_id")
        
        # Evitiamo duplicati se il documento compare più volte
        if chunk_id in processed_chunks:
            continue
            
        titolo, contenuto = estrai_titolo_e_contenuto(doc.page_content)
        
        # Fallback se il titolo non è presente nel testo ma nei metadata
        if not titolo and "titolo_bando" in doc.metadata:
            titolo = doc.metadata["titolo_bando"]

        dataset.append({
            "id": chunk_id,
            "tipo": "chunk",
            "titolo": titolo,
            "contenuto": contenuto,
            "livello": 0,
            "sons": []  # Nessun figlio per le foglie
        })
        processed_chunks.add(chunk_id)

    # ============================================================
    # 2. CREAZIONE NODI SUMMARY (Padri)
    # ============================================================
    print("🔹 Creazione nodi per i Summary e associazione figli...")
    for g_cluster, topics in summaries.items():
        for topic, info in topics.items():
            
            # ID univoco per il summary
            summary_id = f"summary_{g_cluster}_{topic}"
            
            # Estrazione titolo e contenuto del summary
            summary_text = info.get("summary", "")
            titolo, contenuto = estrai_titolo_e_contenuto(summary_text)
            
            # Recupero dei Document figli da local_clusters_chunks
            # (gestisce sia chiavi integer che stringa)
            g_key = int(g_cluster) if isinstance(g_cluster, str) and g_cluster.isdigit() else g_cluster
            t_key = int(topic) if isinstance(topic, str) and topic.isdigit() else topic
            
            figli_docs = local_clusters_chunks.get(g_key, {}).get(t_key, [])
            if not figli_docs:
                # Tentativo di recupero con chiavi stringa
                figli_docs = local_clusters_chunks.get(str(g_cluster), {}).get(str(topic), [])

            # Estrazione degli ID reali dei figli dai metadata dei Document
            sons_ids = [doc.metadata["chunk_id"] for doc in figli_docs if "chunk_id" in doc.metadata]

            dataset.append({
                "id": summary_id,
                "tipo": "summary",
                "titolo": titolo,
                "contenuto": contenuto,
                "livello": 1,
                "sons": sons_ids  # Lista di 'chunk_id' dei documenti contenuti nel cluster
            })


    print(f"✅ Dataset generato con successo! {len(dataset)} elementi totali.")
    return dataset

In [33]:
# Generazione del dataset
dataset_grafo = crea_dataset_grafo(documents, summaries, local_clusters_chunks)

# Verifica del primo chunk e del primo summary
print("Esempio Chunk:", dataset_grafo[0])
print("Esempio Summary:", [row for row in dataset_grafo if row["tipo"] == "summary"][0])

🔹 Creazione nodi per i Chunk originali...
🔹 Creazione nodi per i Summary e associazione figli...
✅ Dataset generato con successo! 1260 elementi totali.
Esempio Chunk: {'id': '1_Finality_0', 'tipo': 'chunk', 'titolo': 'Veneto – Interventi di recupero, promozione e valorizzazione delle aree interne attraverso interventi ad alto impatto culturale - Finality', 'contenuto': 'Il bando è finalizzato a sostenere il miglioramento delle condizioni e della fruibilità del patrimonio pubblico nelle aree interne del Veneto e garantisce il rispetto dei diritti fondamentali e la conformità alla Carta dei diritti fondamentali dell’Unione europea. Il bando agevola interventi ed attività che contribuiscono al raggiungimento dei seguenti obiettivi dell’Agenda 2030 per lo Sviluppo Sostenibile: Lavoro dignitoso e crescita economica; Imprese, innovazione e infrastrutture; Città e Comunità sostenibili; Consumo e produzione responsabili; Lotta contro il cambiamento climatico.', 'livello': 0, 'sons': []}
Esempi

In [38]:
import time
import re
import numpy as np
from tqdm import tqdm
from umap import UMAP
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from transformers import AutoTokenizer
from langchain_huggingface import HuggingFaceEmbeddings
from google import genai
from google.genai import types
from google.genai.errors import ClientError

# ============================================================
# PROMPT DI SISTEMA ED UTENTE
# ============================================================
SYSTEM_PROMPT = """
Sei un assistente specializzato nella sintesi tematica di testi per sistemi RAG gerarchici (RAPTOR).
Il tuo compito NON è riassumere ogni frammento in sequenza, ma identificare i temi, i pattern e i meccanismi comuni che attraversano l'intero gruppo di frammenti, producendo una sintesi concettuale unificata.
Devi essere accurato e non aggiungere mai informazioni non supportate dal contenuto fornito.
Devi attenerti STRETTAMENTE al formato richiesto, senza mai aggiungere introduzioni, commenti, spiegazioni sul tuo operato o frasi di chiusura.
"""

USER_PROMPT_TEMPLATE = """
I seguenti frammenti di testo appartengono a un unico gruppo tematico di bandi/documenti relativi a finanziamenti pubblici.

IL TUO COMPITO:
Genera un titolo generale e un riassunto che catturino il TEMA COMUNE e i PATTERN RICORRENTI condivisi dai frammenti, non un elenco descrittivo bando per bando.

COME STRUTTURARE IL RIASSUNTO:
1. Individua gli elementi trasversali che accomunano i frammenti (beneficiari, meccanismo di incentivo, finalità).
2. Organizza il discorso attorno a QUESTI elementi trasversali, non attorno all'elenco delle singole regioni/enti/bandi.
3. Cita regioni, enti o bandi specifici solo come esempi a supporto di un pattern, non come struttura portante di ogni frase.
4. Se più bandi condividono lo stesso meccanismo, raggruppali in un'unica frase (es. "Diverse regioni, tra cui X, Y e Z, adottano un pagamento annuale per ettaro a compensazione di...") invece di descriverli uno per uno.
5. Il Contenuto deve essere un UNICO blocco di prosa continua e scorrevole. NON usare grassetto, sottotitoli, intestazioni di sezione, o qualsiasi altra suddivisione visiva in "capitoletti" o "aree tematiche" separate. NON usare elenchi puntati o numerati.

COME CHIUDERE IL TESTO (regola più importante):
Il "Contenuto" deve terminare su un dettaglio concreto e specifico — MAI su una frase che riepiloga, generalizza o commenta l'insieme di quanto appena scritto.

COSA EVITARE TASSATIVAMENTE:
- Elencare i frammenti nell'ordine in cui sono presentati.
- NON citare mai l'etichetta numerica del frammento originale (es. "Frammento 3").

REGOLE TASSATIVE SULL'OUTPUT:
- TITOLO: massimo 12 parole.
- LUNGHEZZA MASSIMA DEL CONTENUTO: al massimo 300 parole — nessun sottotitolo, nessun grassetto.

FORMATO RICHIESTO:

Titolo: <titolo specifico e distintivo del cluster>

Contenuto: <riassunto unificato organizzato per pattern tematici, in prosa continua>

Testo completo dei frammenti da sintetizzare:
{context}
"""


# ============================================================
# CLASSE DI GESTIONE CHIAVI API
# ============================================================
class KeyManager:
    def __init__(self, keys):
        self.keys = [k for k in keys if k]
        self.cooldown_until = {k: 0.0 for k in self.keys}
        self.current_index = 0
        self.clients = {k: genai.Client(api_key=k) for k in self.keys}

    def _available_keys(self):
        now = time.time()
        return [k for k in self.keys if self.cooldown_until[k] <= now]

    def mark_rate_limited(self, key, wait_seconds):
        self.cooldown_until[key] = time.time() + wait_seconds

    def get_next_key(self):
        available = self._available_keys()
        if not available:
            return None
        for i in range(len(self.keys)):
            idx = (self.current_index + i) % len(self.keys)
            k = self.keys[idx]
            if k in available:
                self.current_index = (idx + 1) % len(self.keys)
                return k
        return None

    def seconds_until_next_available(self):
        now = time.time()
        return max(0, min(self.cooldown_until[k] for k in self.keys) - now)

    def client_for(self, key):
        return self.clients[key]


# ============================================================
# FUNZIONE PRINCIPALE RAPTOR
# ============================================================
def genera_riassunti_raptor(
    chunks: list,
    gemini_api_keys: list,
    model_id: str = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
    gemini_model_name: str = "gemini-3.6-flash",
    soglia_cumulativa: float = 0.90,
    device: str = "cpu"
) -> list[dict]:
    """
    Esegue la pipeline completa RAPTOR (Clustering Gerarchico UMAP+GMM + Sintesi Gemini).
    
    Parameters:
    - chunks: lista di stringhe o oggetti Document (LangChain)
    - gemini_api_keys: lista di chiavi API per la rotazione
    - model_id: modello HuggingFace per gli embedding
    - gemini_model_name: nome del modello Gemini per i riassunti
    - soglia_cumulativa: soglia per il soft assignment nei cluster GMM
    
    Returns:
    - Lista di dizionari con la seguente struttura:
      [
        {
          "summary": "Titolo: ... \n\nContenuto: ...",
          "child_indices": [0, 3, 7],         # Indici dei chunk figli
          "child_chunks": ["testo 0", ...],   # Testi dei chunk figli
          "tokens": 185                       # Numero di token della sintesi
        },
        ...
      ]
    """
    if not chunks or len(chunks) < 2:
        print("⚠️ Servono almeno 2 chunk per poter eseguire il clustering.")
        return []

    # 1. Normalizzazione input (estrazione testo se oggetti Document)
    raw_texts = [
        c.page_content if hasattr(c, "page_content") else (c if isinstance(c, str) else str(c))
        for c in chunks
    ]
    totale_chunks = len(raw_texts)

    # 2. Inizializzazione modelli e gestori
    key_manager = KeyManager(gemini_api_keys)
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    
    print(f"🔹 Caricamento modello di embedding: {model_id}...")
    hf_embeddings = HuggingFaceEmbeddings(
        model_name=model_id,
        model_kwargs={"device": device},
        encode_kwargs={"normalize_embeddings": True}
    )

    # 3. Vectorizzazione in memoria
    print(f"🔹 Calcolo degli embedding per {totale_chunks} frammenti...")
    embeddings = hf_embeddings.embed_documents(raw_texts)
    X = np.array(embeddings)

    # 4. Riduzione Dimensionale Globale (PCA + UMAP)
    print("🔹 Esecuzione PCA...")
    pca_comp = 0.90 if X.shape[0] > 10 else min(X.shape[0] - 1, X.shape[1])
    pca = PCA(n_components=pca_comp, random_state=42)
    X_pca = pca.fit_transform(X)

    # UMAP Globale (adattamento dinamico di n_neighbors se pochi campioni)
    n_neighbors_global = min(10, max(2, int((len(X_pca) - 1) ** 0.5) + 1))
    umap_dim_global = min(10, max(1, len(X_pca) - 1))
    
    print(f"🔹 Esecuzione UMAP Globale (samples={len(X_pca)}, n_neighbors={n_neighbors_global})...")
    X_umap_global = UMAP(
        n_neighbors=n_neighbors_global,
        n_components=umap_dim_global,
        metric="euclidean",
        random_state=42
    ).fit_transform(X_pca)

    # 5. Helper GMM + BIC
    def find_optimal_gmm(data, max_k=80):
        max_k_calc = min(max_k, len(data))
        if max_k_calc < 2:
            g = GaussianMixture(n_components=1, covariance_type='full', random_state=42).fit(data)
            return 1, g
        
        best_bic, best_k, best_gmm = np.inf, 1, None
        for k in range(1, max_k_calc + 1):
            gmm = GaussianMixture(n_components=k, covariance_type='full', random_state=42, init_params='kmeans')
            gmm.fit(data)
            bic = gmm.bic(data)
            if bic < best_bic:
                best_bic, best_k, best_gmm = bic, k, gmm
        return best_k, best_gmm
# 6. Clustering Globale e Soft Assignment
    optimal_k, global_gmm = find_optimal_gmm(X_umap_global, max_k=min(80, max(2, totale_chunks // 2)))
    global_probs = global_gmm.predict_proba(X_umap_global)

    global_clusters_indices = {i: [] for i in range(optimal_k)}
    for idx, probs in enumerate(global_probs):
        sorted_idx = np.argsort(probs)[::-1]
        cum_probs = np.cumsum(probs[sorted_idx])
        n_to_take = np.searchsorted(cum_probs, soglia_cumulativa) + 1
        for c_id in sorted_idx[:n_to_take]:
            global_clusters_indices[c_id].append(idx)

    # 7. Clustering Locale (UMAP + GMM) per identificare i Topic Finali
    print("🔹 Esecuzione Clustering Locale (Topic Extraction)...")
    final_topics_chunk_indices = []

    for g_id, indices in global_clusters_indices.items():
        if len(indices) < 2:
            if len(indices) == 1:
                final_topics_chunk_indices.append(indices)
            continue

        X_local = X_pca[indices]
        n_samples_local = len(X_local)
        
        # --- FIX UMAP LOCALE: Bypass SOTA e Iperparametri Dinamici ---
        if n_samples_local <= 3:
            # Regola d'oro: con 3 o meno punti non si riduce, si passa l'output PCA diretto al GMM
            X_umap_local = X_local
        else:
            # Calcolo dei limiti basati su N (evita di chiedere a UMAP più di quanto i dati consentano)
            n_neighbors_local = min(10, max(2, n_samples_local - 1))
            umap_dim_local = min(10, max(1, n_samples_local - 2))
            
            X_umap_local = UMAP(
                n_neighbors=n_neighbors_local,
                n_components=umap_dim_local,
                min_dist=0.0,         # SOTA per clustering: massimizza la densità dei punti
                metric="cosine",      # SOTA per embeddings semantici (testo)
                init="random",        # Bypass spettrale: evita il crash "k >= N" di scipy
                random_state=42
            ).fit_transform(X_local)

        # Il calcolo del GMM avviene adesso senza crash e su componenti ben distinte
        max_k_loc = max(2, n_samples_local // 2)
        opt_loc_k, local_gmm = find_optimal_gmm(X_umap_local, max_k=max_k_loc)
        local_probs = local_gmm.predict_proba(X_umap_local)
        topic_dict = {i: [] for i in range(opt_loc_k)}
        for i, probs in enumerate(local_probs):
            sorted_idx = np.argsort(probs)[::-1]
            cum_probs = np.cumsum(probs[sorted_idx])
            n_to_take = np.searchsorted(cum_probs, soglia_cumulativa) + 1
            
            orig_idx = indices[i]
            for t_id in sorted_idx[:n_to_take]:
                topic_dict[t_id].append(orig_idx)

        for t_indices in topic_dict.values():
            if t_indices:
                # Deduplica gli indici mantenendo l'ordine
                unique_indices = list(dict.fromkeys(t_indices))
                final_topics_chunk_indices.append(unique_indices)

    print(f"✅ Trovati {len(final_topics_chunk_indices)} gruppi tematici unici. Avvio generazione con Gemini...")

    # 8. Generazione Riassunti con Gemini
    def parse_wait_seconds(error_str: str, default: int = 60) -> int:
        match_retry = re.search(r'retryDelay["\']?\s*:\s*["\']?(\d+(?:\.\d+)?)s', error_str)
        if match_retry:
            return int(float(match_retry.group(1))) + 5
        return default

    def summarize_context(context_text: str, max_retries: int = 20) -> str:
        prompt = USER_PROMPT_TEMPLATE.format(context=context_text)
        attempt = 0
        while attempt < max_retries:
            current_key = key_manager.get_next_key()
            if current_key is None:
                wait_s = max(1, key_manager.seconds_until_next_available())
                time.sleep(wait_s)
                continue

            attempt += 1
            client = key_manager.client_for(current_key)
            try:
                response = client.models.generate_content(
                    model=gemini_model_name,
                    contents=prompt,
                    config=types.GenerateContentConfig(
                        system_instruction=SYSTEM_PROMPT,
                        temperature=0.0,
                    ),
                )
                return response.text.strip()
            except ClientError as e:
                err_str = str(e)
                if "429" in err_str or "RESOURCE_EXHAUSTED" in err_str:
                    w_sec = parse_wait_seconds(err_str)
                    key_manager.mark_rate_limited(current_key, w_sec)
                else:
                    time.sleep(5)
            except Exception:
                time.sleep(5)
        return "ERRORE_GENERAZIONE: Impossibile generare la sintesi."

    # 9. Costruzione dell'Output Finale
    risultati_finali = []

    for topic_indices in tqdm(final_topics_chunk_indices, desc="Sintesi Topic"):
        # Prepara il testo unificato dei soli chunk appartenenti a questo topic
        context_parts = [
            f"--- Frammento {i+1} ---\n{raw_texts[idx]}"
            for i, idx in enumerate(topic_indices)
        ]
        context_str = "\n\n".join(context_parts)

        # Chiamata a Gemini
        summary_text = summarize_context(context_str)
        
        # Conteggio token
        token_count = len(tokenizer.encode(summary_text))

        # Associa il riassunto agli indici e ai testi dei figli
        risultati_finali.append({
            "summary": summary_text,
            "child_indices": topic_indices,
            "child_chunks": [raw_texts[idx] for idx in topic_indices],
            "tokens": token_count
        })

    print("\n✅ Elaborazione completata con successo!")
    return risultati_finali

In [39]:
# ============================================================
# 1. ISOLIAMO I SUMMARY DI LIVELLO 1 DAL DATASET
# ============================================================
# Prendiamo i summary presenti nel dataset per passarli a RAPTOR
import os
nodi_livello_1 = [nodo for nodo in dataset_grafo if nodo["tipo"] == "summary"]

# Estraiamo i testi nel formato "Titolo: ... \n\n Contenuto: ..."
testi_livello_1 = [
    f"Titolo: {nodo['titolo']}\nContenuto: {nodo['contenuto']}" 
    for nodo in nodi_livello_1
]

print(f"🔹 Invio di {len(testi_livello_1)} summary di Livello 1 a RAPTOR...")

# ============================================================
# 2. ESEGUIAMO UN SINGOLO GIRO DI PIPELINE
# ============================================================
GEMINI_API_KEYS = [
    os.environ.get("GEMINI_API_KEY_1", "AQ.Ab8RN6LzERfVFSHpL1BB7JM-FfTxzvMPdlSs6HdLvZtig6h6Og")
    # aggiungi altre chiavi qui se ne hai altre
]

risultati_livello_2 = genera_riassunti_raptor(
    chunks=testi_livello_1,
    gemini_api_keys= GEMINI_API_KEYS
)

# ============================================================
# 3. AGGIUNGIAMO I NUOVI SUMMARY DI LIVELLO 2 IN CODA AL DATASET
# ============================================================
print(f"🔹 Generati {len(risultati_livello_2)} summary di Livello 2. Aggiunta al dataset...")

for idx, item in enumerate(risultati_livello_2):
    summary_raw = item["summary"]
    titolo, contenuto = estrai_titolo_e_contenuto(summary_raw)
    
    # Mappiamo i figli: child_indices punta alla posizione nella lista nodi_livello_1
    child_indices = item["child_indices"]
    sons_ids = [nodi_livello_1[c_idx]["id"] for c_idx in child_indices]

    # Creiamo la nuova riga per il Livello 2
    nuova_riga_l2 = {
        "id": f"summary_L2_{idx}",
        "livello": 2,
        "tipo": "summary",
        "titolo": titolo,
        "contenuto": contenuto,
        "sons": sons_ids  # <--- Qui ci sono gli ID dei summary di Livello 1!
    }
    
    # Appendiamo semplicemente in fondo
    dataset_grafo.append(nuova_riga_l2)

print(f"✅ Fatto! Dataset aggiornato. Ora contiene {len(dataset_grafo)} nodi totali.")

🔹 Invio di 172 summary di Livello 1 a RAPTOR...
🔹 Caricamento modello di embedding: sentence-transformers/paraphrase-multilingual-mpnet-base-v2...
🔹 Calcolo degli embedding per 172 frammenti...
🔹 Esecuzione PCA...
🔹 Esecuzione UMAP Globale (samples=172, n_neighbors=10)...


c:\Users\Allocca_g\Documents\GRAPH_RAG\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


🔹 Esecuzione Clustering Locale (Topic Extraction)...


c:\Users\Allocca_g\Documents\GRAPH_RAG\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\Allocca_g\Documents\GRAPH_RAG\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\Allocca_g\Documents\GRAPH_RAG\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\Allocca_g\Documents\GRAPH_RAG\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\Allocca_g\Documents\GRAPH_RAG\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\Allocca_g\Documents\GRAPH_RAG\.venv\Lib\site-p

✅ Trovati 30 gruppi tematici unici. Avvio generazione con Gemini...


Sintesi Topic: 100%|██████████| 30/30 [07:08<00:00, 14.27s/it]


✅ Elaborazione completata con successo!
🔹 Generati 30 summary di Livello 2. Aggiunta al dataset...
✅ Fatto! Dataset aggiornato. Ora contiene 1290 nodi totali.


In [45]:
# ============================================================
# 1. ISOLIAMO I SUMMARY DI LIVELLO 2 DAL DATASET
# ============================================================
# Prendiamo SOLO i summary generati nel giro precedente (Livello 2)
nodi_livello_2 = [nodo for nodo in dataset_grafo if nodo.get("livello") == 2]

# Estraiamo i testi nel formato "Titolo: ... \n\n Contenuto: ..."
testi_livello_2 = [
    f"Titolo: {nodo['titolo']}\nContenuto: {nodo['contenuto']}" 
    for nodo in nodi_livello_2
]

print(f"🔹 Invio di {len(testi_livello_2)} summary di Livello 2 a RAPTOR...")

# ============================================================
# 2. ESEGUIAMO UN NUOVO GIRO DI PIPELINE PER IL LIVELLO 3
# ============================================================
GEMINI_API_KEYS = [
    os.environ.get("GEMINI_API_KEY_1", "AQ.Ab8RN6LzERfVFSHpL1BB7JM-FfTxzvMPdlSs6HdLvZtig6h6Og")
    # Puoi mantenere le stesse chiavi
]

risultati_livello_3 = genera_riassunti_raptor(
    chunks=testi_livello_2,
    gemini_api_keys=GEMINI_API_KEYS
)

# ============================================================
# 3. AGGIUNGIAMO I NUOVI SUMMARY DI LIVELLO 3 IN CODA AL DATASET
# ============================================================
print(f"🔹 Generati {len(risultati_livello_3)} summary di Livello 3. Aggiunta al dataset...")

for idx, item in enumerate(risultati_livello_3):
    summary_raw = item["summary"]
    titolo, contenuto = estrai_titolo_e_contenuto(summary_raw)
    
    # Mappiamo i figli: child_indices punta alla posizione nella lista nodi_livello_2
    child_indices = item["child_indices"]
    sons_ids = [nodi_livello_2[c_idx]["id"] for c_idx in child_indices]

    # Creiamo la nuova riga per il Livello 3
    nuova_riga_l3 = {
        "id": f"summary_L3_{idx}",
        "livello": 3,
        "tipo": "summary",
        "titolo": titolo,
        "contenuto": contenuto,
        "sons": sons_ids  # <--- Qui ci sono gli ID dei summary di Livello 2!
    }
    
    # Appendiamo in fondo al grafo principale
    dataset_grafo.append(nuova_riga_l3)

print(f"✅ Fatto! Dataset aggiornato con il Livello 3. Ora contiene {len(dataset_grafo)} nodi totali.")

🔹 Invio di 30 summary di Livello 2 a RAPTOR...
🔹 Caricamento modello di embedding: sentence-transformers/paraphrase-multilingual-mpnet-base-v2...
🔹 Calcolo degli embedding per 30 frammenti...
🔹 Esecuzione PCA...
🔹 Esecuzione UMAP Globale (samples=30, n_neighbors=6)...


c:\Users\Allocca_g\Documents\GRAPH_RAG\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\Allocca_g\Documents\GRAPH_RAG\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\Allocca_g\Documents\GRAPH_RAG\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\Allocca_g\Documents\GRAPH_RAG\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\Allocca_g\Documents\GRAPH_RAG\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\Allocca_g\Documents\GRAPH_RAG\.venv\Lib\site-p

🔹 Esecuzione Clustering Locale (Topic Extraction)...
✅ Trovati 14 gruppi tematici unici. Avvio generazione con Gemini...


Sintesi Topic: 100%|██████████| 14/14 [03:04<00:00, 13.16s/it]


✅ Elaborazione completata con successo!
🔹 Generati 14 summary di Livello 3. Aggiunta al dataset...
✅ Fatto! Dataset aggiornato con il Livello 3. Ora contiene 1304 nodi totali.


In [47]:
# ============================================================
# 1. ISOLIAMO I SUMMARY DI LIVELLO 3 DAL DATASET
# ============================================================
# Prendiamo SOLO i summary generati nel giro precedente (Livello 3)
nodi_livello_3 = [nodo for nodo in dataset_grafo if nodo.get("livello") == 3]

# Estraiamo i testi nel formato "Titolo: ... \n\n Contenuto: ..."
testi_livello_3 = [
    f"Titolo: {nodo['titolo']}\nContenuto: {nodo['contenuto']}" 
    for nodo in nodi_livello_3
]

print(f"🔹 Invio di {len(testi_livello_3)} summary di Livello 3 a RAPTOR...")

# ============================================================
# 2. ESEGUIAMO UN NUOVO GIRO DI PIPELINE PER IL LIVELLO 4
# ============================================================
GEMINI_API_KEYS = [
    os.environ.get("GEMINI_API_KEY_1", "AQ.Ab8RN6LzERfVFSHpL1BB7JM-FfTxzvMPdlSs6HdLvZtig6h6Og")
    # Puoi mantenere le stesse chiavi
]

risultati_livello_4 = genera_riassunti_raptor(
    chunks=testi_livello_3,
    gemini_api_keys=GEMINI_API_KEYS
)

# ============================================================
# 3. AGGIUNGIAMO I NUOVI SUMMARY DI LIVELLO 4 IN CODA AL DATASET
# ============================================================
print(f"🔹 Generati {len(risultati_livello_4)} summary di Livello 4. Aggiunta al dataset...")

for idx, item in enumerate(risultati_livello_4):
    summary_raw = item["summary"]
    titolo, contenuto = estrai_titolo_e_contenuto(summary_raw)
    
    # Mappiamo i figli: child_indices punta alla posizione nella lista nodi_livello_3
    child_indices = item["child_indices"]
    sons_ids = [nodi_livello_3[c_idx]["id"] for c_idx in child_indices]

    # Creiamo la nuova riga per il Livello 4
    nuova_riga_l4 = {
        "id": f"summary_L4_{idx}",
        "livello": 4,
        "tipo": "summary",
        "titolo": titolo,
        "contenuto": contenuto,
        "sons": sons_ids  # <--- Qui ci sono gli ID dei summary di Livello 3!
    }
    
    # Appendiamo in fondo al grafo principale
    dataset_grafo.append(nuova_riga_l4)

print(f"✅ Fatto! Dataset aggiornato con il Livello 4. Ora contiene {len(dataset_grafo)} nodi totali.")

🔹 Invio di 14 summary di Livello 3 a RAPTOR...
🔹 Caricamento modello di embedding: sentence-transformers/paraphrase-multilingual-mpnet-base-v2...
🔹 Calcolo degli embedding per 14 frammenti...


c:\Users\Allocca_g\Documents\GRAPH_RAG\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\Allocca_g\Documents\GRAPH_RAG\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\Allocca_g\Documents\GRAPH_RAG\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\Allocca_g\Documents\GRAPH_RAG\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


🔹 Esecuzione PCA...
🔹 Esecuzione UMAP Globale (samples=14, n_neighbors=4)...
🔹 Esecuzione Clustering Locale (Topic Extraction)...
✅ Trovati 6 gruppi tematici unici. Avvio generazione con Gemini...


Sintesi Topic: 100%|██████████| 6/6 [01:23<00:00, 13.85s/it]


✅ Elaborazione completata con successo!
🔹 Generati 6 summary di Livello 4. Aggiunta al dataset...
✅ Fatto! Dataset aggiornato con il Livello 4. Ora contiene 1310 nodi totali.


In [51]:
from collections import Counter

print(f"🔹 Elementi totali in dataset_grafo: {len(dataset_grafo)}")
counts = Counter([nodo.get("livello", "Sconosciuto") for nodo in dataset_grafo])
print("🔹 Distribuzione per livello in Python:")
for livello, quantita in sorted(counts.items()):
    print(f"   - Livello {livello}: {quantita} nodi")

🔹 Elementi totali in dataset_grafo: 1310
🔹 Distribuzione per livello in Python:
   - Livello 0: 1088 nodi
   - Livello 1: 172 nodi
   - Livello 2: 30 nodi
   - Livello 3: 14 nodi
   - Livello 4: 6 nodi


In [53]:
from neo4j import GraphDatabase

def ingest_raptor_pulito(dataset_grafo, uri, user, password):
    print(f"🔹 Connessione a Neo4j ({uri})...")
    driver = GraphDatabase.driver(uri, auth=(user, password))
    
    with driver.session() as session:
        # 1. Pialliamo tutto per eliminare ogni residuo precedente
        print("🧹 Pulizia totale del database...")
        session.run("MATCH (n) DETACH DELETE n")
        
        # 2. Raggruppamento per livello
        nodi_per_livello = {}
        for row in dataset_grafo:
            lvl = row.get("livello", 0)
            if lvl not in nodi_per_livello:
                nodi_per_livello[lvl] = []
            nodi_per_livello[lvl].append(row)
            
        # 3. Creazione nodi con etichette PURAMENTE specifiche (zero RaptorNode)
        print("🔹 Inserimento nodi con etichette di livello dedicate...")
        for livello, batch in nodi_per_livello.items():
            if livello == 0 or livello is None:
                label = "Chunk"
            else:
                label = f"SummaryL{livello}"
            
            # Usiamo i backticks per sicurezza sull'etichetta dinamica
            query_nodi = f"""
            UNWIND $batch AS row
            CREATE (n:`{label}` {{id: row.id}})
            SET n.livello = row.livello,
                n.tipo = row.tipo,
                n.titolo = row.titolo,
                n.contenuto = row.contenuto
            """
            session.run(query_nodi, batch=batch)
            print(f"   ↳ Inseriti {len(batch)} nodi con etichetta netta: :{label}")
            
        # 4. Creazione relazioni basate unicamente sull'ID globale
        print("🔹 Creazione delle relazioni gerarchiche (:HAS_CHILD)...")
        query_relazioni = """
        UNWIND $batch AS row
        MATCH (parent {id: row.id})
        WHERE row.sons IS NOT NULL AND size(row.sons) > 0
        UNWIND row.sons AS son_id
        MATCH (son {id: son_id})
        MERGE (parent)-[:HAS_CHILD]->(son)
        """
        session.run(query_relazioni, batch=dataset_grafo)
        
    driver.close()
    print("✅ Ingest completato! Database pulito, etichette stratificate e relazioni collegate.")

# ============================================================
# CONFIGURAZIONE ED ESECUZIONE
# ============================================================
NEO4J_URI = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "pepopepo"

ingest_raptor_pulito(dataset_grafo, NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD)

🔹 Connessione a Neo4j (bolt://localhost:7687)...
🧹 Pulizia totale del database...
🔹 Inserimento nodi con etichette di livello dedicate...
   ↳ Inseriti 1088 nodi con etichetta netta: :Chunk
   ↳ Inseriti 172 nodi con etichetta netta: :SummaryL1
   ↳ Inseriti 30 nodi con etichetta netta: :SummaryL2
   ↳ Inseriti 14 nodi con etichetta netta: :SummaryL3
   ↳ Inseriti 6 nodi con etichetta netta: :SummaryL4
🔹 Creazione delle relazioni gerarchiche (:HAS_CHILD)...
✅ Ingest completato! Database pulito, etichette stratificate e relazioni collegate.


In [54]:
from collections import defaultdict

# 1. Creiamo un dizionario di supporto ID -> Nodo per leggere i titoli facilmente
node_dict = {nodo.get("id"): nodo for nodo in dataset_grafo}

print("==================================================")
print(" VERIFICA 1: Topic (Genitori) con più figli")
print("==================================================")
parent_multipli = 0

for nodo in dataset_grafo:
    sons = nodo.get("sons", [])
    if sons and len(sons) > 1:
        parent_multipli += 1
        titolo = nodo.get("titolo", "Senza titolo")
        livello = nodo.get("livello", "N/A")
        print(f"🔹 [Livello {livello}] '{titolo}' ha {len(sons)} figli.")

print(f"\nTotale nodi genitore con più di un figlio: {parent_multipli}\n")


print("==================================================")
print(" VERIFICA 2: Figli con più di un topic (Genitori multipli)")
print("==================================================")
# Macciamo ogni child_id alla lista dei suoi genitori
child_to_parents = defaultdict(list)

for nodo in dataset_grafo:
    sons = nodo.get("sons", [])
    if sons:
        for son_id in sons:
            child_to_parents[son_id].append(nodo)

figli_multipli = 0
for son_id, parents in child_to_parents.items():
    if len(parents) > 1:
        figli_multipli += 1
        figlio_node = node_dict.get(son_id, {})
        titolo_figlio = figlio_node.get("titolo", str(son_id)[:10])
        nomi_genitori = [p.get("titolo", "Senza titolo") for p in parents]
        print(f"🔸 Figlio '{titolo_figlio}' appartiene a {len(parents)} topic differenti: {nomi_genitori}")

print(f"\nTotale figli con più di un genitore: {figli_multipli}")

 VERIFICA 1: Topic (Genitori) con più figli
🔹 [Livello 1] 'Incentivi regionali per la sostenibilità agricola, biodiversità e tutela del territorio' ha 6 figli.
🔹 [Livello 1] 'Finanziamenti per la conservazione dell'agrobiodiversità, emergenze sanitarie zootecniche e benessere animale' ha 7 figli.
🔹 [Livello 1] 'Finanziamenti per la tutela agroforestale, la resilienza climatica e la modernizzazione agricola' ha 11 figli.
🔹 [Livello 1] 'Finanziamenti per la tutela dell agrobiodiversità dell apicoltura e della gestione fitosanitaria' ha 10 figli.
🔹 [Livello 1] 'Incentivi pubblici per pianificazione forestale, aggregazione fondiaria e riforestazione' ha 9 figli.
🔹 [Livello 1] 'Indennità compensative e contributi straordinari per aree montane e svantaggiate' ha 7 figli.
🔹 [Livello 1] 'Finanziamenti per la biosicurezza suinicola, prevenzione della peste suina e benessere animale' ha 6 figli.
🔹 [Livello 1] 'Sostegno e finanziamenti per la tutela degli impollinatori e della biodiversità' ha 7 

In [56]:
texts = []
metadata = []

print("Estrazione dei testi dai nodi del grafo...")
for nodo in dataset_grafo:
    titolo = nodo.get("titolo", "")
    contenuto = nodo.get("contenuto", "")
    
    # Uniamo titolo e contenuto per dare più contesto all'embedding
    testo_completo = f"Titolo: {titolo}\nContenuto: {contenuto}"
    texts.append(testo_completo)
    metadata.append(nodo)

print(f"Trovati {len(texts)} nodi da processare.")

Estrazione dei testi dai nodi del grafo...
Trovati 1310 nodi da processare.


In [57]:
texts

['Titolo: Veneto – Interventi di recupero, promozione e valorizzazione delle aree interne attraverso interventi ad alto impatto culturale - Finality\nContenuto: Il bando è finalizzato a sostenere il miglioramento delle condizioni e della fruibilità del patrimonio pubblico nelle aree interne del Veneto e garantisce il rispetto dei diritti fondamentali e la conformità alla Carta dei diritti fondamentali dell’Unione europea. Il bando agevola interventi ed attività che contribuiscono al raggiungimento dei seguenti obiettivi dell’Agenda 2030 per lo Sviluppo Sostenibile: Lavoro dignitoso e crescita economica; Imprese, innovazione e infrastrutture; Città e Comunità sostenibili; Consumo e produzione responsabili; Lotta contro il cambiamento climatico.',
 'Titolo: Veneto – Bando Musei - Finality\nContenuto: Il bando stabilisce i termini e disciplina criteri di attività proposte dai musei per il miglioramento o il raggiungimento di specifici livelli minimi di servizio. Nello specifico il bando i

In [58]:
from tqdm import tqdm
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

# ============================================================
# 0. Configurazione Iniziale
# ============================================================
# Sostituisci con il modello desiderato (es. multilingua)
model_id = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"

# Assicurati che 'dataset_grafo' sia la tua lista di nodi in memoria

# ============================================================
# 1. Inizializzazione del Modello di Embedding (con normalizzazione)
# ============================================================
print(f"Caricamento del modello {model_id} in memoria...")

hf_embeddings = HuggingFaceEmbeddings(
    model_name=model_id,
    model_kwargs={
        "device": "cpu"
    },
    encode_kwargs={
        "normalize_embeddings": True  # Normalizzazione attiva nativamente
    }
)

# Gestione sicura della lunghezza massima di sequenza
if hasattr(hf_embeddings, "client"):
    hf_embeddings.client.max_seq_length = 512
elif hasattr(hf_embeddings, "embedding_ctx"):
    hf_embeddings.embedding_ctx.max_seq_length = 512
elif hasattr(hf_embeddings, "_client"):
    hf_embeddings._client.max_seq_length = 512

# ============================================================
# 2. Preparazione dei Documenti LangChain dai Nodi del Grafo
# ============================================================
print("Estrazione dei testi dai nodi del grafo e conversione in Documenti...")
documents = []

for nodo in dataset_grafo:
    titolo = nodo.get("titolo", "")
    contenuto = nodo.get("contenuto", "")
    
    # Uniamo titolo e contenuto per dare più contesto all'embedding
    testo_completo = f"Titolo: {titolo}\nContenuto: {contenuto}"
    
    # Creiamo il Documento LangChain associando il testo e l'intero nodo come metadati
    doc = Document(
        page_content=testo_completo,
        metadata=nodo
    )
    documents.append(doc)

totale_chunks = len(documents)
print(f"Trovati {totale_chunks} nodi/documenti da processare.")

# ============================================================
# 3. Configurazione dei Batch
# ============================================================
batch_size = 250  
print(f"\nInizio vettorizzazione in batch da {batch_size} documenti...")

# ============================================================
# 4. Creazione del Database (Primo Batch) e Aggiunta Incrementale
# ============================================================
primo_batch = documents[:batch_size]
vectorstore = FAISS.from_documents(primo_batch, hf_embeddings)

# Aggiunta dei batch successivi con barra di caricamento
for i in tqdm(range(batch_size, totale_chunks, batch_size), desc="Vettorizzazione", unit="batch"):
    batch_corrente = documents[i : i + batch_size]
    vectorstore.add_documents(batch_corrente)

# ============================================================
# 5. Salvataggio su Disco
# ============================================================
cartella_salvataggio = "faiss_index_all_texts"
vectorstore.save_local(cartella_salvataggio)

print(f"\n✅ Operazione completata! Indice e metadati salvati correttamente in: '{cartella_salvataggio}'")

Caricamento del modello sentence-transformers/paraphrase-multilingual-mpnet-base-v2 in memoria...
Estrazione dei testi dai nodi del grafo e conversione in Documenti...
Trovati 1310 nodi/documenti da processare.

Inizio vettorizzazione in batch da 250 documenti...


Vettorizzazione: 100%|██████████| 5/5 [05:06<00:00, 61.26s/batch]


✅ Operazione completata! Indice e metadati salvati correttamente in: 'faiss_index_all_texts'


In [63]:
import time
import numpy as np
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# ============================================================
# 1. Caricamento del Modello e del Database FAISS salvato in precedenza
# ============================================================
print("Caricamento risorse per il retrieval...")
model_id = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"

hf_embeddings = HuggingFaceEmbeddings(
    model_name=model_id,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

cartella_salvataggio = "faiss_index_all_texts"
# Carichiamo il vectorstore salvato con LangChain
vectorstore = FAISS.load_local(cartella_salvataggio, hf_embeddings, allow_dangerous_deserialization=True)

# Estraiamo tutti i metadati (i nodi del grafo) associati ai vettori in FAISS
metadata = [doc.metadata for doc in vectorstore.docstore._dict.values()]

print(f"Recuperati {len(metadata)} documenti dal database FAISS.\n")

# Definiamo la query di test
query = "Quali sono gli incentivi per la transizione ecologica e le energie rinnovabili?"
print(f"\nQuery di test: '{query}'\n")

top_k = 25

# =====================================================================
# FLAT RETRIEVAL (Ricerca vettoriale secca su tutto)
# =====================================================================
start_flat = time.time()

# Usiamo la ricerca nativa di LangChain/FAISS (restituisce tuple di (Document, score))
docs_scores_flat = vectorstore.similarity_search_with_score(query, k=top_k)
flat_results = [doc.metadata for doc, score in docs_scores_flat]

end_flat = time.time()
time_flat = (end_flat - start_flat) * 1000  # in millisecondi

# =====================================================================
# STAMPA DEI RISULTATI E DEI TEMPI
# =====================================================================
print("=" * 60)
print(" REPORT: FLAT RETRIEVAL")
print("=" * 60)
print(f"⏱️  Tempo di esecuzione: {time_flat:.4f} ms")

print("\n" + "=" * 60)
print(" RISULTATI FLAT RETRIEVAL (Top 10)")
print("=" * 60)

for i, res in enumerate(flat_results, 1):
    titolo = res.get('titolo', 'Senza titolo')
    livello = res.get('livello', 'Chunk')
    contenuto = res.get('contenuto', '')
    # Mostra i primi 100 caratteri del contenuto per dare un'idea
    contenuto_preview = contenuto[:150] + "..." if len(contenuto) > 150 else contenuto
    
    print(f"\n{i}. [{livello}] {titolo}")
    print(f"   Score: {docs_scores_flat[i-1][1]:.4f}")
    print(f"   Contenuto: {contenuto_preview}")
    print(f"   ID: {res.get('id', 'N/A')}")

print("\n" + "=" * 60)
print(f"📊 Totale documenti recuperati: {len(flat_results)}")
print("=" * 60)

# Opzionale: stampa tutti i livelli trovati per avere un'idea della composizione
livelli_trovati = {}
for res in flat_results:
    livello = res.get('livello', 'Chunk')
    livelli_trovati[livello] = livelli_trovati.get(livello, 0) + 1

print("\n📈 Composizione dei risultati per livello:")
for livello, count in livelli_trovati.items():
    print(f"   {livello}: {count} documenti")

Caricamento risorse per il retrieval...
Recuperati 1310 documenti dal database FAISS.


Query di test: 'Quali sono gli incentivi per la transizione ecologica e le energie rinnovabili?'

 REPORT: FLAT RETRIEVAL
⏱️  Tempo di esecuzione: 61.4121 ms

 RISULTATI FLAT RETRIEVAL (Top 10)

1. [0] RENEWFM - Meccanismo di finanziamento per le energie rinnovabili specifico per tecnologia - Finality
   Score: 0.4620
   Contenuto: L’obiettivo del bando è contribuire alla transizione verso l’energia pulita e agli obiettivi del Clean Industrial Deal, inclusi i traguardi e gli obie...
   ID: 548_Finality_0

2. [0] Horizon Europe - Prossima generazione di tecnologie per le energie rinnovabili - Finality
   Score: 0.4740
   Contenuto: Il bando rientra nella Destinazione “ Approvvigionamento energetico sostenibile, sicuro e competitivo ” del WP Horizon Europe. Questa destinazione com...
   ID: 910_Finality_0

3. [1] Incentivi e finanziamenti per transizione energetica, batterie, efficienza ed ecomobilità

In [64]:
import time
import numpy as np
from typing import List, Tuple, Dict, Any
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from sentence_transformers import CrossEncoder
from ollama import Client
from datetime import datetime

class RetrievalRerankGenerate:
    def __init__(
        self,
        faiss_path: str = "faiss_index_all_texts",
        embedding_model: str = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
        reranker_model: str = "BAAI/bge-reranker-v2-m3",
        ollama_host: str = "10.70.50.140",
        ollama_model: str = "qwen2.5:14b-instruct",
        temperature: float = 0.0
    ):
        """Inizializza il sistema con i modelli e il database."""
        print("🔄 Inizializzazione del sistema...")
        
        # 1. Embedding model
        self.hf_embeddings = HuggingFaceEmbeddings(
            model_name=embedding_model,
            model_kwargs={"device": "cpu"},
            encode_kwargs={"normalize_embeddings": True}
        )
        
        # 2. FAISS vectorstore
        self.vectorstore = FAISS.load_local(
            faiss_path, 
            self.hf_embeddings, 
            allow_dangerous_deserialization=True
        )
        print(f"✅ Vectorstore caricato: {self.vectorstore.index.ntotal} documenti")
        
        # 3. Reranker
        print(f"🔄 Caricamento reranker {reranker_model}...")
        self.reranker = CrossEncoder(reranker_model)
        print("✅ Reranker caricato")
        
        # 4. Ollama client
        self.ollama_client = Client(host=ollama_host)
        self.ollama_model = ollama_model
        self.temperature = temperature
        print(f"✅ Ollama client configurato su {ollama_host}")
        print(f"   Modello: {ollama_model}")
        print(f"   Temperature: {temperature}")
        
    def retrieve(
        self, 
        query: str, 
        retrieval_k: int = 100
    ) -> Tuple[List[Tuple[Any, float]], float]:
        """
        Fase 1: Retrieval vettoriale.
        
        Args:
            query: La query dell'utente
            retrieval_k: Numero di documenti da recuperare (default: 100)
            
        Returns:
            Tuple (documenti_con_punteggi, tempo_impiegato)
        """
        start_time = time.time()
        
        # Recupera i documenti con i loro punteggi di similarità
        docs_with_scores = self.vectorstore.similarity_search_with_score(
            query, 
            k=retrieval_k
        )
        
        elapsed_time = (time.time() - start_time) * 1000  # millisecondi
        
        return docs_with_scores, elapsed_time
    
    def rerank(
        self,
        query: str,
        docs_with_scores: List[Tuple[Any, float]],
        rerank_k: int = 10
    ) -> Tuple[List[Tuple[Any, float]], float]:
        """
        Fase 2: Reranking dei documenti.
        
        Args:
            query: La query dell'utente
            docs_with_scores: Documenti dal retrieval con i loro punteggi originali
            rerank_k: Numero di documenti da passare al reranker (default: 10)
            
        Returns:
            Tuple (documenti_riordinati_con_punteggi_reranker, tempo_impiegato)
        """
        start_time = time.time()
        
        # Prendi solo i top N documenti (quelli che andranno al reranker)
        top_docs = docs_with_scores[:rerank_k]
        
        # Prepara le coppie (query, documento) per il reranker
        pairs = [(query, doc.page_content) for doc, _ in top_docs]
        
        # Calcola i nuovi punteggi
        rerank_scores = self.reranker.predict(pairs)
        
        # Combina documenti con i nuovi punteggi
        reranked_results = [
            (doc, float(score)) 
            for (doc, _), score in zip(top_docs, rerank_scores)
        ]
        
        # Ordina per punteggio decrescente
        reranked_results.sort(key=lambda x: x[1], reverse=True)
        
        elapsed_time = (time.time() - start_time) * 1000  # millisecondi
        
        return reranked_results, elapsed_time
    
    def generate(
        self,
        query: str,
        reranked_docs: List[Tuple[Any, float]],
        system_prompt: str = None
    ) -> Tuple[str, float]:
        """
        Fase 3: Generazione della risposta con Ollama.
        
        Args:
            query: La query dell'utente
            reranked_docs: Documenti riordinati dal reranker
            system_prompt: Prompt di sistema opzionale
            
        Returns:
            Tuple (risposta_generata, tempo_impiegato)
        """
        start_time = time.time()
        
        # Costruisci il contesto dai documenti
        context_parts = []
        for i, (doc, score) in enumerate(reranked_docs, 1):
            titolo = doc.metadata.get('titolo', 'Senza titolo')
            livello = doc.metadata.get('livello', 'Chunk')
            contenuto = doc.page_content
            
            context_parts.append(
                f"[Documento {i}] - {livello}: {titolo}\n"
                f"Contenuto: {contenuto}\n"
                f"Rilevanza: {score:.4f}\n"
            )
        
        context = "\n".join(context_parts)
        
        # Prompt di sistema predefinito se non fornito
        if system_prompt is None:
            system_prompt = (
                "Sei un assistente esperto che risponde a domande basandosi esclusivamente "
                "sul contesto fornito. Usa solo le informazioni presenti nei documenti per "
                "rispondere. Se non trovi la risposta nel contesto, dillo chiaramente."
            )
        
        # Costruisci il messaggio completo
        user_message = f"""
DOMANDA: {query}

CONTESTO:
{context}

Istruzioni: Rispondi alla domanda basandoti SOLO sul contesto fornito. 
Se la risposta non è presente nel contesto, dillo esplicitamente.
"""
        
        # Richiesta a Ollama con temperature=0
        full_response = ""
        try:
            stream = self.ollama_client.chat(
                model=self.ollama_model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_message}
                ],
                options={
                    "temperature": self.temperature,  # Temperature impostata a 0
                },
                stream=True,
            )
            
            for chunk in stream:
                full_response += chunk["message"]["content"]
                
        except Exception as e:
            full_response = f"❌ Errore durante la generazione: {e}"
        
        elapsed_time = (time.time() - start_time) * 1000  # millisecondi
        
        return full_response, elapsed_time
    
    def process_query(
        self,
        query: str,
        retrieval_k: int = 100,
        rerank_k: int = 10,
        system_prompt: str = None,
        verbose: bool = True
    ) -> Dict[str, Any]:
        """
        Esegue l'intero pipeline: retrieval → reranking → generazione.
        
        Args:
            query: La query dell'utente
            retrieval_k: Numero documenti da recuperare (default: 100)
            rerank_k: Numero documenti da passare al reranker (default: 10)
            system_prompt: Prompt di sistema per Ollama
            verbose: Se True, stampa i dettagli
            
        Returns:
            Dizionario con tutti i risultati e i tempi
        """
        if verbose:
            print("\n" + "=" * 70)
            print(f"🔍 PROCESSAMENTO QUERY")
            print(f"📝 Query: {query}")
            print(f"⏰ Inizio: {datetime.now().strftime('%H:%M:%S.%f')[:-3]}")
            print("=" * 70)
            print(f"\n📊 Configurazione:")
            print(f"   • Retrieval K: {retrieval_k}")
            print(f"   • Rerank K: {rerank_k}")
            print(f"   • Modello Ollama: {self.ollama_model}")
            print(f"   • Temperature: {self.temperature}")
        
        # FASE 1: RETRIEVAL
        if verbose:
            print("\n🔎 Fase 1: Retrieval vettoriale...")
        retrieved_docs, time_retrieval = self.retrieve(query, retrieval_k)
        if verbose:
            print(f"   ✅ Recuperati {len(retrieved_docs)} documenti in {time_retrieval:.2f} ms")
        
        # FASE 2: RERANKING
        if verbose:
            print(f"\n🔄 Fase 2: Reranking (su top {rerank_k} documenti)...")
        reranked_docs, time_rerank = self.rerank(query, retrieved_docs, rerank_k)
        if verbose:
            print(f"   ✅ Riorganizzati {len(reranked_docs)} documenti in {time_rerank:.2f} ms")
        
        # FASE 3: GENERAZIONE
        if verbose:
            print(f"\n🤖 Fase 3: Generazione risposta con Ollama...")
        response, time_generation = self.generate(query, reranked_docs, system_prompt)
        if verbose:
            print(f"   ✅ Risposta generata in {time_generation:.2f} ms")
        
        # Calcolo tempo totale
        total_time = time_retrieval + time_rerank + time_generation
        
        # Risultati
        result = {
            "query": query,
            "retrieval_time": time_retrieval,
            "rerank_time": time_rerank,
            "generation_time": time_generation,
            "total_time": total_time,
            "retrieved_count": len(retrieved_docs),
            "reranked_count": len(reranked_docs),
            "retrieved_docs": retrieved_docs,
            "reranked_docs": reranked_docs,
            "response": response
        }
        
        # Stampa finale riassuntiva
        if verbose:
            print("\n" + "=" * 70)
            print("📊 RIEPILOGO TEMPI")
            print("=" * 70)
            print(f"⏱️  Retrieval:    {time_retrieval:.2f} ms  ({time_retrieval/total_time*100:.1f}%)")
            print(f"⏱️  Reranking:    {time_rerank:.2f} ms  ({time_rerank/total_time*100:.1f}%)")
            print(f"⏱️  Generazione:  {time_generation:.2f} ms  ({time_generation/total_time*100:.1f}%)")
            print(f"⏱️  TOTALE:       {total_time:.2f} ms")
            print("=" * 70)
            print("\n💬 RISPOSTA FINALE:")
            print("-" * 70)
            print(response)
            print("-" * 70)
        
        return result



In [67]:

    # Inizializza il sistema con qwen2.5:14b-instruct e temperature=0
rag_system = RetrievalRerankGenerate(
    faiss_path="faiss_index_all_texts",
    embedding_model="sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
    reranker_model="BAAI/bge-reranker-v2-m3",
    ollama_host="10.70.50.140",
    ollama_model="qwen2.5:14b-instruct",  # Modello aggiornato
    temperature=0.0  # Temperature a 0 per risposte deterministiche
    )
    
    # Query di test
query = "Esiste qualche bando per i musei?"
    
    # Esegui il processo completo
result = rag_system.process_query(
        query=query,
        retrieval_k=30,    # Recupera 100 documenti
        rerank_k=10,        # Rerank i top 10
        system_prompt=None,  # Usa il prompt di default
        verbose=True
    )
    
    # Puoi anche accedere ai risultati singolarmente
print("\n📋 Risultati dettagliati:")
print(f"Query: {result['query']}")
print(f"Tempo totale: {result['total_time']:.2f} ms")
print(f"Documenti recuperati: {result['retrieved_count']}")
print(f"Documenti rerankati: {result['reranked_count']}")
    
    # Mostra i top documenti dopo il reranking
print("\n📄 Top documenti dopo reranking:")
for i, (doc, score) in enumerate(result['reranked_docs'][:5], 1):
    titolo = doc.metadata.get('titolo', 'Senza titolo')
    livello = doc.metadata.get('livello', 'Chunk')
    print(f"{i}. [{livello}] {titolo} (Score: {score:.4f})")


🔄 Inizializzazione del sistema...
✅ Vectorstore caricato: 1310 documenti
🔄 Caricamento reranker BAAI/bge-reranker-v2-m3...
✅ Reranker caricato
✅ Ollama client configurato su 10.70.50.140
   Modello: qwen2.5:14b-instruct
   Temperature: 0.0

🔍 PROCESSAMENTO QUERY
📝 Query: Esiste qualche bando per i musei?
⏰ Inizio: 17:05:54.570

📊 Configurazione:
   • Retrieval K: 30
   • Rerank K: 10
   • Modello Ollama: qwen2.5:14b-instruct
   • Temperature: 0.0

🔎 Fase 1: Retrieval vettoriale...
   ✅ Recuperati 30 documenti in 62.92 ms

🔄 Fase 2: Reranking (su top 10 documenti)...
   ✅ Riorganizzati 10 documenti in 10164.10 ms

🤖 Fase 3: Generazione risposta con Ollama...
   ✅ Risposta generata in 5105.01 ms

📊 RIEPILOGO TEMPI
⏱️  Retrieval:    62.92 ms  (0.4%)
⏱️  Reranking:    10164.10 ms  (66.3%)
⏱️  Generazione:  5105.01 ms  (33.3%)
⏱️  TOTALE:       15332.03 ms

💬 RISPOSTA FINALE:
----------------------------------------------------------------------
Sì, esistono bandi per i musei menzionati nel